# Movies QA -- Inference, Evaluation & Contrastive Training

This notebook runs the full HalluLens pipeline on Movies QA in **closed-book mode**
(no supporting passages provided -- the model relies on parametric memory).

**Dataset:** Movies QA from [LLMsKnow](https://github.com/kaistAI/knowledge_probing) -- actor-role-movie trivia questions.
- `test` split: 7,856 questions (default -- good for eval and training the contrastive model)
- `train` split: 10,000 questions (available for large-scale runs)
- Format: "Who acted as {role} in the movie '{title}'?" with a single actor name answer

**Why movie trivia matters for hallucination detection:**
Pop-culture factual recall tests a different knowledge domain from Wikipedia-based benchmarks
(HotpotQA, NQ). Generalisation across knowledge domains is important for robust hallucination
detection.

**Steps:**
1. Inference -- generate model responses with activation logging
2. Evaluation -- substring match against gold answer; write `eval_results.json`
3. Inspection -- per-sample view and evaluator comparison

**Prerequisite:** A running vLLM server (`COMPUTE_CONTEXT=REMOTE_GPU` in `.env`).

## 1 -- Configuration

In [ ]:
import os, sys, json
from pathlib import Path

# -- Repo root on path --
repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.chdir(repo_root)

# -- Model --
MODEL = "meta-llama/Llama-3.1-8B-Instruct"
model_name = MODEL.split("/")[-1]

# -- Dataset split and sample count --
SPLIT = "test"   # "test" | "train"
N     = None      # None = entire split

# -- Storage paths --
base_dir          = Path("shared/movies")
activations_path  = str(base_dir / "activations.zarr")
log_file          = str(base_dir / "server.log")

output_dir        = "output"
task_dir          = Path(output_dir) / "movies" / model_name
generations_file  = str(task_dir / "generation.jsonl")
eval_results_file     = str(task_dir / "eval_results.json")
eval_results_llm_file = str(task_dir / "eval_results_llm.json")

# -- Activation logging --
LOGGER_TYPE = "zarr"   # zarr (preferred) | lmdb | json

# -- Inference settings --
MAX_TOKENS  = 64
TEMPERATURE = 0.0

# -- Batched inference (ModelAdapter path) --
# Setting BATCH_SIZE > 0 uses HFTransformersAdapter.infer_batch() directly,
# bypassing the HTTP server for higher throughput via batched model.generate().
# Set to None or 0 to use the original sequential server path.
BATCH_SIZE = 8

# -- LLM evaluator --
# Used for the LLM-judge eval (eval_llm step).
# Default: neuralmagic-ent/Llama-3.3-70B-Instruct-quantized.w8a8
LLM_EVALUATOR = None   # None -> use class default

base_dir.mkdir(parents=True, exist_ok=True)
task_dir.mkdir(parents=True, exist_ok=True)

print(f"Model             : {MODEL}")
print(f"Split             : {SPLIT}  (N={N or 'all'})")
print(f"Batch size        : {BATCH_SIZE or 'disabled (sequential server path)'}")
print(f"Generations file  : {generations_file}")
print(f"Eval (substring)  : {eval_results_file}")
print(f"Eval (LLM judge)  : {eval_results_llm_file}")
print(f"Activations       : {activations_path}")
print(f"Logger type       : {LOGGER_TYPE}")

## 2 -- Inference

Generates model responses for Movies QA questions in closed-book mode and logs activations.
Resume-safe: questions already in `generation.jsonl` are skipped.

When `BATCH_SIZE` is set, inference uses `HFTransformersAdapter.infer_batch()` directly --
multiple prompts are packed into a single `model.generate()` call for higher GPU throughput.
No HTTP server is needed in this mode. Activations are written to Zarr asynchronously.

In [ ]:
from scripts.run_with_server import run_experiment

run_experiment(
    step="inference",
    task="movies",
    model=MODEL,
    split=SPLIT,
    N=N,
    logger_type=LOGGER_TYPE,
    activations_path=activations_path,
    log_file=log_file,
    output_dir=output_dir,
    generations_file_path=generations_file,
    max_inference_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    resume=True,
    batch_size=BATCH_SIZE,
)

## 3 -- Evaluation

Two evaluation methods are run and compared:

- **Substring match** (`eval`) -- fast, deterministic. Correct if the gold answer appears verbatim in the response. Saved to `eval_results.json`.
- **LLM judge** (`eval_llm`) -- uses a large language model to assess semantic correctness. Handles alternative name spellings and paraphrased answers. Saved to `eval_results_llm.json`, which also includes substring results for direct comparison.

The LLM eval is resume-safe (incremental saving to `llm_halu_eval_raw.jsonl`).

In [ ]:
from scripts.run_with_server import run_experiment

# -- 3a: Substring match eval --
run_experiment(
    step="eval",
    task="movies",
    model=MODEL,
    output_dir=output_dir,
    generations_file_path=generations_file,
    eval_results_path=eval_results_file,
)

# -- 3b: LLM-judge eval --
run_experiment(
    step="eval_llm",
    task="movies",
    model=MODEL,
    output_dir=output_dir,
    generations_file_path=generations_file,
    eval_results_path=eval_results_llm_file,
    llm_evaluator=LLM_EVALUATOR,
    resume=True,
)

## 4 -- Results Inspection

In [ ]:
import pandas as pd

with open(eval_results_file) as f:
    substr_res = json.load(f)

with open(eval_results_llm_file) as f:
    llm_res = json.load(f)

n = substr_res["total_count"]
llm_correct_count = llm_res["accurate_count"]
substr_correct_count = substr_res["accurate_count"]
agree_count = int(llm_res["evaluator_agreement_rate"] * n)

print(f"Model          : {model_name}")
print(f"Split          : {SPLIT}  (N={n})")
print()
print(f"{'Metric':<30} {'Substring':>12} {'LLM Judge':>12}")
print("-" * 55)
print(f"{'Correct count':<30} {substr_correct_count:>12} {llm_correct_count:>12}")
print(f"{'Correct rate':<30} {substr_res['correct_rate']:>12.1%} {llm_res['correct_rate']:>12.1%}")
print(f"{'Hallucination rate':<30} {substr_res['halu_Rate']:>12.1%} {llm_res['halu_Rate']:>12.1%}")
print("-" * 55)
print(f"{'Evaluator agreement':<30} {llm_res['evaluator_agreement_rate']:>12.1%}")
print()
print(f"LLM evaluator  : {llm_res['evaluator_hallucination']}")

In [ ]:
# -- Evaluator disagreements (where methods diverge) --
# These are the most informative cases: substring says wrong, LLM says correct.
# Indicates substring is under-counting correct answers (e.g., nickname vs full name).
raw_df = pd.read_json(task_dir / "raw_eval_res.jsonl", lines=True)

# Merge LLM judge verdicts
raw_df["halu_llm"]    = llm_res["halu_test_res"]
raw_df["halu_substr"] = llm_res["substring_halu_test_res"]
raw_df["agree"] = raw_df["halu_llm"] == raw_df["halu_substr"]

false_halu = raw_df[raw_df["halu_substr"] & ~raw_df["halu_llm"]].reset_index(drop=True)
false_corr = raw_df[~raw_df["halu_substr"] & raw_df["halu_llm"]].reset_index(drop=True)

print(f"Total disagreements : {(~raw_df['agree']).sum()}  ({(~raw_df['agree']).mean():.1%})")
print(f"  Substr=halu, LLM=correct (likely false hallucinations) : {len(false_halu)}")
print(f"  Substr=correct, LLM=halu (LLM over-penalising)         : {len(false_corr)}")

print("\n-- Top examples: Substring says hallucinated, LLM says correct --")
for i, row in false_halu.head(6).iterrows():
    print(f"\n  Q  : {row['question']}")
    print(f"  Gen: {row['generation']}")
    print(f"  Ans: {row['answer']}")